In [4]:
import pandas as pd

# 1. Load the main WDI data file
print("Loading WDICSV.csv file...")
# Using low_memory=False to prevent data type warnings on large CSVs
raw_df = pd.read_csv("WDICSV.csv", low_memory=False)

# 2. Define target health indicator codes & clean names
health_indicators = {
    'SP.DYN.LE00.IN': 'Life Expectancy (Years)',
    'SH.XPD.CHEX.GD.ZS': 'Health Expenditure (% of GDP)',
    'SH.DTH.MORT': 'Under-5 Mortality Rate (per 1,000)',
    'SH.IMM.MEAS': 'Measles Immunization (%)',
    'SH.MED.NUMW.P3': 'Physicians (per 1,000 people)'
}

# 3. Filter strictly for health indicators
df_health = raw_df[raw_df['Indicator Code'].isin(health_indicators.keys())].copy()

# Add a readable short name column
df_health['Indicator Clean Name'] = df_health['Indicator Code'].map(health_indicators)

# 4. Unpivot year columns (2000 to 2023) into a single Year column
year_cols = [str(year) for year in range(2000, 2024)]
id_cols = ['Country Name', 'Country Code', 'Indicator Clean Name', 'Indicator Code']

df_melted = pd.melt(
    df_health,
    id_vars=id_cols,
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
)

# 5. Clean missing records & data types
df_melted['Year'] = pd.to_numeric(df_melted['Year'])
df_melted['Value'] = pd.to_numeric(df_melted['Value'], errors='coerce')
df_cleaned = df_melted.dropna(subset=['Value']).copy()

# 6. Save the lean dataset for Power BI
output_filename = "WDI_Healthcare_Cleaned.csv"
df_cleaned.to_csv(output_filename, index=False)

print(f"Done! Clean file saved as '{output_filename}' with {len(df_cleaned):,} rows.")

Loading WDICSV.csv file...
Done! Clean file saved as 'WDI_Healthcare_Cleaned.csv' with 26,567 rows.
